# Benchmarking Tabular Explainer
To benchmark `shapiq`'s `TabularExplainer` we run it against the equivalent explainers of the shap library. For comparing different approximators and imputer strategies We run both, the `shapiq` and the `shap` versions against the exact calculation of shapley values.

In [ ]:
from __future__ import annotations

import numpy as np
import shap
import shapiq

## Preparing Datasets and models
We use the `california_housing` and the `bike_sharing` datasets for the benchmark as they are comparably small datasets (so the exact shapley values can be computed in reasonable time) still they are realistic (real world) data sets.
The values are scaled to a range from -1 to 1 and split up in training and test data (80:20).
As both data sets have non-classified output, we use a regression model (scikit-learn's`DecisionTreeRegressor`) to be explained in the benchmark. We use scikit-learn's  as model.

In [ ]:
bike_tree = shapiq.games.benchmark.setup.GameBenchmarkSetup(
    dataset_name="bike_sharing", model_name="decision_tree"
)
house_tree = shapiq.games.benchmark.setup.GameBenchmarkSetup(
    dataset_name="california_housing", model_name="decision_tree"
)
bike_x_train = bike_tree.x_train
bike_x_test = bike_tree.x_test
bike_y_train = bike_tree.y_train
bike_y_test = bike_tree.y_test

house_x_train = house_tree.x_train
house_x_test = house_tree.x_test
house_y_train = house_tree.y_train
house_y_test = house_tree.y_test

##Imputation
As the performance concerning speed as well as quality of an approximator depends strongly on the used imputer, we are going to use different imputers for comparison.
On the other hand the performance of an approximator depends on the number of passes so we are going to run the test on the same amount for each approximator.


In [ ]:
sample_size = 100
# The number of samples to draw from the conditional background data for the imputation.

conditional_budget = np.arange(2, 12, 1)
# The number of coalitions to sample per each point in `data` for training the generative model.

conditional_threshold = 10
# Quantile threshold defining a neighbourhood of samples to draw `sample_size` from.

random_state = 42

test_point_budget = 3

## Running the Explainers


In [ ]:
kern_house = shapiq.approximator.KernelSHAP(len(house_x_train[0]))
kern_bike = shapiq.approximator.KernelSHAP(len(bike_x_train[0]))

svarm_house = shapiq.approximator.SVARM(
    n=len(house_x_train[0]), index="SV", random_state=random_state
)
svarm_bike = shapiq.approximator.SVARM(
    n=len(bike_x_train[0]), index="SV", random_state=random_state
)

perm_house = shapiq.approximator.PermutationSamplingSV(
    n=len(house_x_train[0]), random_state=random_state
)
perm_bike = shapiq.approximator.PermutationSamplingSV(
    n=len(bike_x_train[0]), random_state=random_state
)

house_base = shapiq.TabularExplainer(
    model=house_tree.model,
    data=house_tree.x_train,
    class_index=None,
    imputer="marginal",
    approximator=kern_house,
    index="SV",
    max_order=1,
    random_state=random_state,
    verbose=True,
    sample_size=10 * sample_size,
    joint_marginal_distribution=False,
    normalize=False,
)
bike_base = shapiq.TabularExplainer(
    model=bike_tree.model,
    data=bike_tree.x_train,
    class_index=None,
    imputer="marginal",
    approximator=kern_bike,
    index="SV",
    max_order=1,
    random_state=random_state,
    verbose=True,
    sample_size=10 * sample_size,
    joint_marginal_distribution=False,
    normalize=False,
)

house_tree_kernel = shapiq.TabularExplainer(
    model=house_tree.model,
    data=house_tree.x_train,
    class_index=None,
    imputer="marginal",
    approximator=kern_house,
    index="SV",
    max_order=1,
    random_state=random_state,
    verbose=True,
    sample_size=sample_size,
    joint_marginal_distribution=False,
    normalize=False,
)
bike_tree_kernel = shapiq.TabularExplainer(
    model=bike_tree.model,
    data=bike_tree.x_train,
    class_index=None,
    imputer="marginal",
    approximator=kern_bike,
    index="SV",
    max_order=1,
    random_state=random_state,
    verbose=True,
    sample_size=sample_size,
    joint_marginal_distribution=False,
    normalize=False,
)
house_tree_svarm = shapiq.TabularExplainer(
    model=house_tree.model,
    data=house_tree.x_train,
    class_index=None,
    imputer="marginal",
    approximator=svarm_house,
    index="SV",
    max_order=1,
    random_state=random_state,
    verbose=True,
    sample_size=sample_size,
    joint_marginal_distribution=False,
    normalize=False,
)
bike_tree_svarm = shapiq.TabularExplainer(
    model=bike_tree.model,
    data=bike_tree.x_train,
    class_index=None,
    imputer="marginal",
    approximator=svarm_bike,
    index="SV",
    max_order=1,
    random_state=random_state,
    verbose=True,
    sample_size=sample_size,
    joint_marginal_distribution=False,
    normalize=False,
)

house_tree_perm = shapiq.TabularExplainer(
    model=house_tree.model,
    data=house_tree.x_train,
    class_index=None,
    imputer="marginal",
    approximator=perm_house,
    index="SV",
    max_order=1,
    random_state=random_state,
    verbose=True,
    sample_size=sample_size,
    joint_marginal_distribution=False,
    normalize=False,
)

bike_tree_perm = shapiq.TabularExplainer(
    model=bike_tree.model,
    data=bike_tree.x_train,
    class_index=None,
    imputer="marginal",
    approximator=perm_bike,
    index="SV",
    max_order=1,
    random_state=random_state,
    verbose=True,
    sample_size=sample_size,
    joint_marginal_distribution=False,
    normalize=False,
)

# Accumulators:

h_base_results = []
b_base_results = []

h_shap_kern_results = []
h_shap_perm_results = []
b_shap_kern_results = []
b_shap_perm_results = []

h_shapiq_kern_results = []
h_shapiq_svarm_results = []
h_shapiq_perm_results = []
b_shapiq_kern_results = []
b_shapiq_svarm_results = []
b_shapiq_perm_results = []

for k in range(len(conditional_budget)):
    house_tree_shap_kern = shap.explainers.Kernel(
        model=house_tree.predict_function,
        data=shap.sample(house_tree.x_train, sample_size),
        masker=shap.maskers.Independent(
            data=shap.sample(house_tree.x_train, sample_size), max_samples=conditional_budget[k]
        ),
    )
    bike_tree_shap_kern = shap.explainers.Kernel(
        model=bike_tree.predict_function,
        data=shap.sample(bike_tree.x_train, sample_size),
        masker=shap.maskers.Independent(
            data=shap.sample(bike_tree.x_train, sample_size), max_samples=conditional_budget[k]
        ),
    )

    house_tree_shap_perm = shap.explainers.Permutation(
        model=house_tree.predict_function,
        masker=shap.maskers.Independent(
            data=shap.sample(house_tree.x_train, sample_size), max_samples=conditional_budget[k]
        ),
    )

    bike_tree_shap_perm = shap.explainers.Permutation(
        model=bike_tree.predict_function,
        masker=shap.maskers.Independent(
            data=shap.sample(bike_tree.x_train, sample_size), max_samples=conditional_budget[k]
        ),
    )

    h_t_s_p_expl = house_tree_shap_perm(house_x_test[:test_point_budget])
    h_shap_perm_results.append(h_t_s_p_expl)

    b_t_s_p_expl = bike_tree_shap_perm(bike_x_test[:test_point_budget])
    b_shap_perm_results.append(b_t_s_p_expl)

    h_t_s_k_expl = house_tree_shap_kern(house_x_test[:test_point_budget])
    h_shap_kern_results.append(h_t_s_k_expl)

    b_t_s_k_expl = bike_tree_shap_kern(bike_x_test[:test_point_budget])
    b_shap_kern_results.append(b_t_s_k_expl)

for j in range(test_point_budget):
    h_base_expl = house_base.explain_function(
        x=house_x_test[j], budget=1000, random_state=random_state
    )
    h_base_results.append(h_base_expl)
    b_base_expl = bike_base.explain_function(
        x=bike_x_test[j], budget=1000, random_state=random_state
    )
    b_base_results.append(b_base_expl)

for i in range(len(conditional_budget)):
    for j in range(test_point_budget):
        h_t_k_expl = house_tree_kernel.explain_function(
            x=house_x_test[j], budget=conditional_budget[i], random_state=random_state
        )
        h_shapiq_kern_results.append(h_t_k_expl)

        b_t_k_expl = bike_tree_kernel.explain_function(
            x=bike_x_test[j], budget=conditional_budget[i], random_state=random_state
        )
        b_shapiq_kern_results.append(b_t_k_expl)

        h_t_s_expl = house_tree_svarm.explain_function(
            x=house_x_test[j], budget=conditional_budget[i], random_state=random_state
        )
        h_shapiq_svarm_results.append(h_t_s_expl)

        b_t_s_expl = bike_tree_svarm.explain_function(
            x=bike_x_test[j], budget=conditional_budget[i], random_state=random_state
        )
        b_shapiq_svarm_results.append(b_t_s_expl)

        h_t_p_expl = house_tree_perm.explain_function(
            x=house_x_test[j], budget=conditional_budget[i], random_state=random_state
        )
        h_shapiq_perm_results.append(h_t_p_expl)

        b_t_p_expl = bike_tree_perm.explain_function(
            x=bike_x_test[j], budget=conditional_budget[i], random_state=random_state
        )
        b_shapiq_perm_results.append(b_t_p_expl)

In [ ]:
display(h_base_results)
# display(h_shap_kern_results)
for t in range(test_point_budget):
    display(h_base_results[t].values)
for s in range(len(conditional_budget)):
    display(h_shapiq_kern_results[s * 3])
    display(h_shapiq_kern_results[s * 3].values)

## Calculating the L1 and L2 Difference to Baseline

In [ ]:
# Function to calculate the average L1 Difference of shap explanations where b is the shapiq baseline explanation
def __shap_average_l1_diffs(x, b) -> np.array:
    sums_of_budgets = np.zeros(len(conditional_budget))
    for o in range(len(conditional_budget)):
        for m in range(test_point_budget):
            sum_all_points = 0
            for n in range(len(house_x_test[0])):
                diff_point = (abs(abs((b[m]).values[n + 1]) - abs((x[o]).values[m, n]))) / len(
                    house_x_test[0]
                )
                sum_all_points = sum_all_points + diff_point
            sum_budget = sum_all_points / test_point_budget
        sums_of_budgets[o] = sum_budget
    return sums_of_budgets


# Function to calculate the average L1 Difference of shapiq explanations where b is the shapiq baseline explanation
def __shapiq_average_l1_diffs(x, b) -> np.array:
    sums_of_budgets = np.zeros(len(conditional_budget))
    for o in range(len(conditional_budget)):
        for m in range(test_point_budget):
            sum_all_points = 0
            for n in range(len(house_x_test[0])):
                diff_point = (
                    abs(
                        abs((b[m]).values[n + 1])
                        - abs((x[(o - 1) * test_point_budget + m]).values[n + 1])
                    )
                ) / len(house_x_test[0])
                sum_all_points = sum_all_points + diff_point
            sum_budget = sum_all_points / test_point_budget
        sums_of_budgets[o] = sum_budget
    return sums_of_budgets


h_shap_kern_average_l1 = __shap_average_l1_diffs(x=h_shap_kern_results, b=h_base_results)
display(h_shap_kern_average_l1)

h_shapiq_kern_average_l1 = __shapiq_average_l1_diffs(x=h_shapiq_kern_results, b=h_base_results)
display(h_shapiq_kern_average_l1)

display(h_base_results)

h_shapiq_svarm_average_l1 = __shapiq_average_l1_diffs(x=h_shapiq_svarm_results, b=h_base_results)
display(h_shapiq_svarm_average_l1)

h_shapiq_perm_average_l1 = __shapiq_average_l1_diffs(x=h_shapiq_perm_results, b=h_base_results)
display(h_shapiq_perm_average_l1)

# h_base_results
# b_base_results

# h_shap_kern_results
# h_shap_perm_results
# b_shap_kern_results
# b_shap_perm_results

# h_shapiq_kern_results
# h_shapiq_svarm_results
# h_shapiq_perm_results
# b_shapiq_kern_results
# b_shapiq_svarm_results
# b_shapiq_perm_results

In [ ]:
display(h_shap_kern_results)
display(h_shap_perm_results)
display(h_shapiq_kern_results)
display(h_shapiq_svarm_results)
display(h_shapiq_perm_results)